# Curs 2 — Ecosistemul de modele

Scopul acestui notebook: testăm **2-3 modele diferite** pe același input și alegem modelul potrivit pentru proiect.

Vom folosi:
1. **Gemini** — providerul principal, prin cheia obținută din Google AI Studio.
2. **OpenRouter** — provider alternativ, util pentru comparație și backup când Gemini are limite de quota.
## OpenRouter — de unde luăm cheia
1. Intră pe https://openrouter.ai/
2. Creează cont sau autentifică-te.
3. Mergi la **Keys**.
4. Creează un nou API key.
5. Copiază cheia în fișierul `.env`:
```env
OPENROUTER_API_KEY=pune-cheia-ta-aici
---

In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import json

## 1. Configurare — mai multe modele

In [2]:
MODELE = [
    ("gemini", "gemini-2.5-flash-lite", "Gemini 2.5 Flash Lite"),
    ("gemini", "gemini-2.5-flash", "Gemini 2.5 Flash"),
    ("openrouter", "openrouter/free", "OpenRouter Free"),
]

print("Modele pregătite:", [nume for _, _, nume in MODELE])

Modele pregătite: ['Gemini 2.5 Flash Lite', 'Gemini 2.5 Flash', 'OpenRouter Free']


In [3]:
# Configurăm providerii și cheile API din fișierul .env

load_dotenv()

BASE_URLS = {
    "gemini": "https://generativelanguage.googleapis.com/v1beta/openai/",
    "openrouter": "https://openrouter.ai/api/v1"
}

API_KEYS = {
    "gemini": os.getenv("GEMINI_API_KEY"),
    "openrouter": os.getenv("OPENROUTER_API_KEY")
}

def make_client(provider):
    """Creează clientul API pentru providerul ales."""
    return OpenAI(
        api_key=API_KEYS[provider],
        base_url=BASE_URLS[provider]
    )

## 2. Funcție helper — trimitem același prompt la orice model

În loc să scriem același cod de 3 ori, facem o funcție.

In [4]:
# varianta minimala

# fara functie
client = make_client("gemini")
prompt = "Explică în 2 propoziții ce este FC Barcelona."
response = client.chat.completions.create(
    model="gemini-2.5-flash-lite",
    messages=[
        {"role": "user", "content": prompt}
    ]
)
print(response.choices[0].message.content)

# cu functie
def ask(provider, model, prompt):
    client = make_client(provider)

    messages = [
        {"role": "user", "content": prompt}
    ]
    response = client.chat.completions.create(
        model=model,
        messages=messages
    )
    return response.choices[0].message.content

# iar functia poate fi apelata astfel:
raspuns = ask(
    provider="gemini",
    model="gemini-2.5-flash-lite",
    prompt="Explică în 2 propoziții ce este un LLM."
)

print(raspuns)

FC Barcelona este un club de fotbal profesionist spaniol, renumit la nivel mondial pentru stilul său de joc ofensiv și pentru succesul obținut pe plan intern și internațional. Este unul dintre cele mai populare și valoroase cluburi de sport din lume, cu o istorie bogată și o bază imensă de fani.
Un LLM (Large Language Model) este un tip de inteligență artificială antrenat pe o cantitate masivă de text, permițându-i să înțeleagă, să genereze și să răspundă la limbajul uman într-un mod fluent și coerent. Prin învățarea modelelor și a relațiilor din datele textuale, aceste modele pot efectua o varietate largă de sarcini legate de limbaj, cum ar fi traducerea, rezumarea sau scrierea de conținut creativ.


In [5]:
from openai import RateLimitError, APIError, AuthenticationError
import json

def ask(provider, model, prompt, system=None, temperature=0.7, json_schema=None):
    """Trimite un prompt la model. Poate returna text simplu sau JSON structurat."""

    client = make_client(provider)

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    extra_args = {}

    if json_schema:
        extra_args["response_format"] = {
            "type": "json_schema",
            "json_schema": json_schema
        }

    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=temperature,
            **extra_args
        )

        text = response.choices[0].message.content.strip()

        if json_schema:
            return json.loads(text)

        return text

    except RateLimitError:
        return f"[Eroare: quota/rate limit pentru modelul {model}.]"

    except AuthenticationError:
        return "[Eroare: API key invalidă sau lipsă. Verifică .env.]"

    except APIError as e:
        return f"[Eroare API: {e}]"

    except Exception as e:
        return f"[Eroare: {type(e).__name__} — {e}]"

## 3. Test 1 — Calitatea pe limba română

Testăm dacă modelele înțeleg și răspund corect în română.

In [6]:
PROMPT_RO = """
Acționează ca un analist politic pro-european, ferm anti-suveranist. 
Rezumă în exact 2 propoziții scurte, în română, principalele schimbări din politica românească din ultimii 5 ani. 
Critică ascensiunea populismului naționalist și subliniază nevoia de a menține strânse legăturile cu Uniunea Europeană și Occidentul. 
Maximum 80 de cuvinte.
"""

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    raspuns = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT_RO,
        temperature=0.2
    )

    print(raspuns)


--- Gemini 2.5 Flash Lite ---
În ultimii cinci ani, politica românească a fost marcată de o polarizare crescândă și de o retorică suveranistă, subminând progresele democratice. Ascensiunea populismului naționalist amenință stabilitatea și prosperitatea, subliniind urgența consolidării legăturilor cu Uniunea Europeană și Occidentul pentru a asigura un viitor european.

--- Gemini 2.5 Flash ---
Ultimii cinci ani au adus o consolidare a partidelor tradiționale într-o coaliție largă. Simultan, am asistat la ascensiunea îngrijorătoare a forțelor naționalist-populiste. Această tendință este profund dăunătoare și subminează valorile democratice. Este imperativ să respingem discursurile divizive și să consolidăm legăturile cu Uniunea Europeană și partenerii occidentali, singura cale spre prosperitate și stabilitate.

--- OpenRouter Free ---
În ultimii 5 ani, principala schimbare în politica românească a fost ascensiunea alarmantă a populismului naționalist suveranist, cu retorici anti-Uniunea

## 4. Test 2 — Urmează instrucțiunile din system prompt+ adnotare

Vedem dacă modelele respectă rolul dat prin `system`.

In [7]:
SYSTEM = """
Ești un analist politic pro-european și ferm anti-suveranist. 
Rolul tău este să analizezi comentariile politice, sancționând derapajele populiste, naționalismul extrem și izolaționismul. 
Răspunzi concis, incisiv și interpretezi textul apărând valorile occidentale."""

PROMPT = """
Analizează următorul comentariu politic:
"Toți politicienii fură, iar oamenii simpli plătesc nota. Nimeni nu mai ascultă poporul."

Răspunde în 4 linii:
Ton:
Emoție dominantă:
Țintă principală:
Populism: da/nu
"""

for provider, model, name in MODELE:
    print("\n---", name, "---")
    print(ask(
        provider=provider,
        model=model,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0
    ))


--- Gemini 2.5 Flash Lite ---
Tonul este cinic și acuzator. Emoția dominantă este frustrarea și neîncrederea. Ținta principală este clasa politică, prezentată ca fiind coruptă și deconectată de realitatea cetățenilor. Populismul este prezent, prin generalizarea negativă a politicienilor și apelul la "popor" ca victimă.

--- Gemini 2.5 Flash ---
Ton: Acuzator, fatalist.
Emoție dominantă: Resentiment, neîncredere.
Țintă principală: Elita politică și, implicit, sistemul democratic.
Populism: da.

--- OpenRouter Free ---
**Ton:** Acuzator, despectiv  
**Emoție dominantă:** Frică / Frustrare  
**Țintă principală:** Poporul român  
**Populism:** da


## 5. Test 3 — Output structurat (JSON)

Agenții noștri vor trebui să returneze date structurate.
Testăm dacă modelele pot produce JSON valid la cerere.

In [8]:
SCHEMA_ADNOTARE = {
    "name": "adnotare_comentariu_politic",
    "schema": {
        "type": "object",
        "properties": {
            "ton": {
                "type": "string",
                "enum": ["pozitiv", "negativ", "neutru"]
            },
            "emotie_dominanta": {
                "type": "string",
                "enum": ["furie", "frica", "speranta", "dezamagire", "ironie", "dispret", "neutru"]
            },
            "tinta_principala": {
                "type": "string"
            },
            "populism": {
                "type": "boolean"
            },
            "atac_la_persoana": {
                "type": "boolean"
            },
            "explicatie_scurta": {
                "type": "string"
            }
        },
        "required": [
            "ton",
            "emotie_dominanta",
            "tinta_principala",
            "populism",
            "atac_la_persoana",
            "explicatie_scurta"
        ],
        "additionalProperties": False
    }
}

In [9]:
COMENTARIU = "Acest discurs anti-european este doar o perdea de fum a corupților care se ascund sub steagul patriotismului. Izolarea de partenerii occidentali ne va distruge economia."

SYSTEM = """Ești un asistent de cercetare pro-european și anti-suveranist. 
Rolul tău este să adnotezi comentarii politice, analizând critic discursul naționalist, izolaționist sau anti-occidental. 
Trebuie să fii atent la nuanțele de dispreț, la retorica populistă și la atacurile la persoană. 
Răspunzi obiectiv pe baza grilei, dar din perspectiva apărării valorilor democratice occidentale."""

PROMPT = f"Adnotează următorul comentariu politic: {COMENTARIU}"

for provider, model_id, nume in MODELE:
    print("\n---", nume, "---")

    rezultat = ask(
        provider=provider,
        model=model_id,
        prompt=PROMPT,
        system=SYSTEM,
        temperature=0.1,
        json_schema=SCHEMA_ADNOTARE
    )

    print(rezultat)


--- Gemini 2.5 Flash Lite ---
{'ton': 'negativ', 'emotie_dominanta': 'dispret', 'tinta_principala': 'coruptii si discursul anti-european', 'populism': True, 'atac_la_persoana': False, 'explicatie_scurta': "Comentariul critică discursul anti-european, asociindu-l cu corupția și avertizând asupra consecințelor economice negative ale izolării de partenerii occidentali. Utilizează o retorică populistă prin contrastul 'corupți vs. patrioți'."}

--- Gemini 2.5 Flash ---
{'ton': 'negativ', 'emotie_dominanta': 'dispret', 'tinta_principala': 'discursul anti-european și corupții care se ascund sub steagul patriotismului', 'populism': False, 'atac_la_persoana': True, 'explicatie_scurta': 'Comentariul critică vehement discursul anti-european, considerându-l o tactică de diversiune a unor indivizi corupți. Avertizează asupra consecințelor economice negative ale izolării de partenerii occidentali, apărând valorile pro-europene și parteneriatele strategice.'}

--- OpenRouter Free ---
[Eroare: TypeEr

## 6. Test 4 — Stabilitate la temperature diferite

Un model bun pentru agenți trebuie să fie **stabil** — același input, răspunsuri similare.
Testăm cu Gemini (poți schimba cu orice model).

In [10]:
PROMPT_STAB = """
Curtea Constituțională a anulat alegerile.
Explică în exact 2 propoziții ce poate însemna acest haos instituțional pentru direcția pro-europeană a țării.
Răspunde din perspectiva unui analist anti-suveranist, avertizând asupra modului în care populiștii pot exploata această criză democratică.
"""

TEMPERATURI = [0.1, 0.7, 1.2]

print("[ Test 4 — stabilitate: același prompt, temperaturi diferite ]")

for provider, model_id, nume in MODELE:
    print("\n" + "=" * 60)
    print(f"[ {nume} ]")

    for temp in TEMPERATURI:
        raspuns = ask(
            provider=provider,
            model=model_id,
            prompt=PROMPT_STAB,
            temperature=temp
        )

        print(f"\ntemperature={temp}:")
        print(raspuns)

[ Test 4 — stabilitate: același prompt, temperaturi diferite ]

[ Gemini 2.5 Flash Lite ]

temperature=0.1:
Anularea alegerilor de către Curtea Constituțională, deși un act de control constituțional, poate fi exploatată de forțele populiste pentru a submina încrederea în instituțiile democratice și a promova narative anti-europene, prezentând situația ca pe o dovadă a instabilității și corupției sistemului. Această criză instituțională, alimentată de retorica suveranistă, riscă să izoleze țara pe plan internațional și să încetinească procesul de integrare europeană, creând un teren fertil pentru discursuri naționaliste și eurosceptice.

temperature=0.7:
Anularea alegerilor de către Curtea Constituțională, deși menită să corecteze nereguli, poate fi percepută ca o subminare a voinței populare, oferind populiștilor o pârghie pentru a discredita instituțiile democratice și a promova narative anti-europene bazate pe ideea unei elite deconectate. Această criză instituțională, exploatată abi

## 7. Alegerea modelului pentru proiect

Completați tabelul după testele de mai sus. Nu căutați „cel mai bun model” în general, ci modelul cel mai potrivit pentru proiectul vostru.
| Model | Răspunde bine în română? | Respectă instrucțiunile? | Merge pentru adnotare? | Are erori / quota? | Observație scurtă |
|---|---|---|---|---|---|
| Gemini 2.5 Flash Lite | da | da | da  |  nu | Foarte stabil, insa repeta ideea de baza  |
| OpenRouter Free | parțial | da | parțial | nu  | Are cateva greseli gramaticale |
| Gemini 2.5 Flash | da | da | da | nu | Cel mai calitativ |
### Decizie
**Model principal ales:**  Gemini 2.5 Flash

**Model de rezervă:**  Gemini 2.5 Flash Lite

**Temperature recomandată:** 0.2

**De ce am ales acest model?**  
Am ales Gemini 2.5 Flash ca model principal deoarece a dovedit cea mai înaltă calitate a răspunsurilor în limba română, păstrând o coerență excelentă a textului.

## 8. Configurația finală a proiectului

putem să copiem asta in core/config.py

In [12]:
# core/config.py
# Configurația modelului ales de echipă după testele din Cursul 2.
# Nu puneți chei API aici. Cheile rămân doar în fișierul local .env.
PROVIDER_PRINCIPAL = "gemini"
MODEL_PRINCIPAL = "gemini-2.5-flash"
PROVIDER_FALLBACK = "gemini"
MODEL_FALLBACK = "gemini-2.5-flash-lite"
TEMPERATURE = 0.2

---

## Livrabile C2

Până la cursul următor:

- [ ] Notebook completat cu 2-3 modele testate
- [ ] Matricea de decizie completată cu observații reale
- [ ] README actualizat cu modelul ales și justificarea
- [ ] `.env` configurat cu cheia pentru modelul ales